# 05 -- Suitability and Neighbourhoods

Weighted suitability scoring, focal smoothing, morphological cleanup, and
distance fields.  This notebook draws from command-line examples 22 and 25.

**Run the individual scripts:** `22_map_algebra_suitability.py`,
`25_map_algebra_focal.py`

## Setup

In [ ]:
import sys, os
from pathlib import Path

def _repo_root():
    """Find the Lunarscout repository root from the kernel working directory."""
    for start in [Path.cwd()] + list(Path.cwd().parents):
        if (start / "src" / "lunarscout" / "__init__.py").exists():
            return start
    raise RuntimeError(
        "Cannot locate Lunarscout repository root. "
        "Launch Jupyter from the repository root directory."
    )

_REPO = _repo_root()
sys.path.insert(0, str(_REPO / "src"))
sys.path.insert(0, str(_REPO / "examples"))


import lunarscout as ls
import numpy as np

ma = ls.map_algebra

from _example_support import (
    ensure_synthetic_scenario, ensure_synthetic_series,
    synthetic_georef, synthetic_dem,
)

WORKSPACE = Path("/tmp/lunarscout_notebook_05")
WORKSPACE.mkdir(parents=True, exist_ok=True)

ensure_synthetic_scenario(WORKSPACE)
georef = synthetic_georef()
SCENARIO = ls.open_scenario(WORKSPACE / "synthetic_scenario")

# Load DEM and temporal data as eager rasters.
dem, dem_georef = ls.read_geotiff(SCENARIO.dem_path())
ls.require_same_grid(georef, dem_georef)

slope_vals, _ = ls.slope(dem, dem_georef, output_nodata=-9999.0)
slope = ma.raster(slope_vals, georef, units="deg", name="slope")

series = ensure_synthetic_series(WORKSPACE)
illum_mean, _ = ls.temporal_mean(series)
sun = ma.from_existing(illum_mean, georef, units="fraction", name="illumination_mean")

print(f"Slope raster:  {slope}")
print(f"Sun raster:    {sun}")
print(f"Workspace:     {WORKSPACE}")

---
## 1. Weighted Suitability

Construct hard constraints, normalise scores, and combine them with
explicit weights.  Use `where` to preserve scores only for candidate cells.

**Thresholds are illustrative, not mission recommendations.**

In [ ]:
# --- Configuration (edit these) ---
SLOPE_MAX = 8.0          # degrees
SUN_MIN = 0.60           # mean fraction
WEIGHT_SUN = 0.4
WEIGHT_SLOPE = 0.6
# ---------------------------------

# Hard constraints.
accept_slope = slope <= SLOPE_MAX
accept_sun = sun >= SUN_MIN
candidate = accept_slope & accept_sun

print(f"Candidate cells: {candidate.values.sum()} / {candidate.values.size}")

# Normalised scores (0 = worst, 1 = best).
slope_norm = ma.normalize_minmax(slope, minimum=0.0, maximum=15.0)
sun_norm = ma.normalize_minmax(sun, minimum=0.0, maximum=1.0)

score = (WEIGHT_SUN * sun_norm) + (WEIGHT_SLOPE * (1.0 - slope_norm))
scored = ma.where(candidate, score, ma.invalid)
scored = scored.with_name("weighted_score")

# Print statistics for valid cells.
stats = ma.statistics(scored)
print(f"\nScore statistics (valid cells only):")
print(f"  min:    {stats.min_val}")
print(f"  max:    {stats.max_val}")
print(f"  mean:   {stats.mean}")
print(f"  std:    {stats.std}")
print(f"  count:  {stats.count}")
print(f"  invalid: {stats.invalid_count}")

In [ ]:
_suit = SCENARIO.output_path("analysis/screening")
_suit.mkdir(parents=True, exist_ok=True)

nodata_gref = georef.with_nodata(0)
candidate_out = _suit / "candidate.tif"
ls.write_geotiff(
    str(candidate_out),
    candidate.values.astype("uint8"),
    nodata_gref,
    overwrite=True,
)
print(f"Candidate mask: {candidate_out}")

score_out = _suit / "candidate_score.tif"
ma.write(
    str(score_out),
    scored.expression(),
    dtype=np.float32,
    invalid_value=-9999.0,
    overwrite=True,
)
print(f"Score raster:   {score_out}")

**Try this:** change the weights so sun counts more than slope.  Change
`SLOPE_MAX` to 10.0 and observe how the candidate area grows.  What happens
if you set `SUN_MIN` to 1.0?

---
## 2. Focal Smoothing

A 3x3 focal mean smooths the slope raster.  This demonstrates
neighbourhood operations.

In [ ]:
smoothed = ma.focal_mean(slope, size=3, edge="nearest")
print(f"Original slope  range: [{slope.values.min():.1f}, {slope.values.max():.1f}]")
print(f"Smoothed slope  range: [{smoothed.values.min():.1f}, {smoothed.values.max():.1f}]")

STEEP = 8.0
steep = smoothed >= STEEP
print(f"\nSteep cells (>= {STEEP} deg, smoothed): {steep.values.sum()}")

---
## 3. Morphological Opening

Remove small isolated steep features with morphological opening.

In [ ]:
opened = ma.opening(steep, size=3)
print(f"Before opening: {steep.values.sum()} steep cells")
print(f"After opening:  {opened.values.sum()} steep cells")

---
## 4. Distance to a Feature

Compute the Euclidean distance from every cell to the nearest steep-zone
pixel.

In [ ]:
dist = ma.distance_to(opened, metric="euclidean", units="pixels")
d_stats = ma.statistics(dist)
print(f"Distance statistics ({dist.units}):")
print(f"  min:   {d_stats.min_val}")
print(f"  max:   {d_stats.max_val}")
print(f"  mean:  {d_stats.mean}")
print(f"  count: {d_stats.count}")

In [ ]:
_focal = SCENARIO.output_path("analysis/focal")
_focal.mkdir(parents=True, exist_ok=True)

ls.write_geotiff(str(_focal / "smoothed_slope.tif"), smoothed.values, georef, overwrite=True)
ls.write_geotiff(str(_focal / "opened_mask.tif"), opened.values.astype("uint8"), nodata_gref, overwrite=True)
ls.write_geotiff(str(_focal / "distance_to_steep.tif"), dist.values, georef, overwrite=True)
print(f"Focal products written under {_focal}")

**Try this:** compute distance in physical metres instead of pixels.
What additional information does Lunarscout need to support that?

---
## Cleanup

In [ ]:
series.close()